# TMJ Heatmap Detector — DataSphere Training

**Self-contained notebook** — no repo import required.

Pipeline:
1. Install deps, set paths
2. Download & extract pre-processed volumes (~63 MB) + annotations (~1 MB)
3. Define model, dataset, loss inline
4. Train 3D U-Net, save best checkpoint to filestore

## 1. Setup

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'tqdm', 'scipy'])
print('deps ok')

In [ ]:
import os, sys, json, random, logging, datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy import ndimage
from tqdm.auto import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────
FILESTORE    = Path('/home/jupyter/filestore')          # persistent storage
DATA_DIR     = FILESTORE / 'heatmap_data'
VOLUMES_DIR  = DATA_DIR / 'heatmap_volumes'             # .npy uint8 volumes
ANN_DIR      = DATA_DIR / 'roi_annotations'             # *_rois.json
SPLIT_JSON   = DATA_DIR / 'detector_split.json'
EXP_DIR      = FILESTORE / 'experiments'

DATA_DIR.mkdir(parents=True, exist_ok=True)
EXP_DIR.mkdir(parents=True, exist_ok=True)

# ── Device ─────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'PyTorch: {torch.__version__}')

# numpy compat fix for PyTorch 2.0.x + numpy 2.x
if not hasattr(np, 'core'):
    import numpy.core
    sys.modules.setdefault('numpy._core', numpy.core)
elif not hasattr(np.core, 'multiarray'):
    sys.modules.setdefault('numpy._core', np.core)

## 2. Download data

In [ ]:
import urllib.request, tarfile

RELEASE_BASE = 'https://github.com/tzopiz/MasterProject/releases/download'

def download_release_asset(tag: str, filename: str, dest: Path):
    if dest.exists():
        print(f'  Already exists: {dest}')
        return
    url = f'{RELEASE_BASE}/{tag}/{filename}'
    print(f'  Downloading {url} ...')
    tmp = dest.parent / (dest.name + '.tmp')
    urllib.request.urlretrieve(url, tmp)
    tmp.rename(dest)
    print(f'  Saved → {dest} ({dest.stat().st_size/1e6:.1f} MB)')


# ── Volumes (~63 MB) ───────────────────────────────────────────────────────
vol_tar = DATA_DIR / 'heatmap_volumes.tar.gz'
download_release_asset('heatmap-volumes-v1', 'heatmap_volumes.tar.gz', vol_tar)

if not VOLUMES_DIR.exists() or not any(VOLUMES_DIR.glob('*.npy')):
    print('Extracting volumes...')
    with tarfile.open(vol_tar) as tf:
        tf.extractall(DATA_DIR)
    npy_count = len(list(VOLUMES_DIR.glob('*.npy')))
    print(f'Extracted {npy_count} volumes → {VOLUMES_DIR}')
else:
    print(f'Volumes already extracted: {len(list(VOLUMES_DIR.glob("*.npy")))} files')

# ── Annotations (~200 KB) ─────────────────────────────────────────────────
ann_tar = DATA_DIR / 'roi_annotations.tar.gz'
download_release_asset('annotations-v1', 'roi_annotations.tar.gz', ann_tar)

if not ANN_DIR.exists() or not any(ANN_DIR.glob('*.json')):
    print('Extracting annotations...')
    with tarfile.open(ann_tar) as tf:
        tf.extractall(DATA_DIR)
    n = len(list(ANN_DIR.glob('*.json')))
    print(f'Extracted {n} annotations → {ANN_DIR}')
else:
    print(f'Annotations already present: {len(list(ANN_DIR.glob("*.json")))} files')

# ── Split JSON ─────────────────────────────────────────────────────────────
split_src = DATA_DIR / 'detector_split.json'
download_release_asset('annotations-v1', 'detector_split.json', split_src)
if not SPLIT_JSON.exists():
    split_src.rename(SPLIT_JSON)

with open(SPLIT_JSON) as f:
    split = json.load(f)
print(f"Split — train:{len(split['train'])}  val:{len(split['val'])}  test:{len(split['test'])}")

## 3. Model

In [ ]:
from typing import List, Optional, Tuple


def _double_conv(in_ch: int, out_ch: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
        nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
    )


class _EncoderBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.conv = _double_conv(in_ch, out_ch)
        self.pool = nn.MaxPool3d(2)

    def forward(self, x):
        skip = self.conv(x)
        return self.pool(skip), skip


class _DecoderBlock(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int):
        super().__init__()
        self.up   = nn.ConvTranspose3d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = _double_conv(in_ch // 2 + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.pad(x, [0, skip.shape[4]-x.shape[4],
                           0, skip.shape[3]-x.shape[3],
                           0, skip.shape[2]-x.shape[2]])
        return self.conv(torch.cat([skip, x], dim=1))


class TMJHeatmapDetector(nn.Module):
    """3D U-Net: (B,1,D,H,W) → (B,2,D,H,W) raw logits."""

    def __init__(self, in_channels: int = 1,
                 features: Optional[List[int]] = None,
                 out_channels: int = 2):
        super().__init__()
        if features is None:
            features = [32, 64, 128, 256]

        self.encoders = nn.ModuleList()
        prev = in_channels
        for f in features:
            self.encoders.append(_EncoderBlock(prev, f))
            prev = f

        self.bottleneck = _double_conv(features[-1], features[-1] * 2)
        prev = features[-1] * 2

        self.decoders = nn.ModuleList()
        for f in reversed(features):
            self.decoders.append(_DecoderBlock(prev, f, f))
            prev = f

        self.head = nn.Conv3d(features[0], out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        skips = []
        for enc in self.encoders:
            x, skip = enc(x)
            skips.append(skip)
        x = self.bottleneck(x)
        for dec, skip in zip(self.decoders, reversed(skips)):
            x = dec(x, skip)
        return self.head(x)


model = TMJHeatmapDetector().to(device)
n = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n/1e6:.2f}M')

## 4. Dataset

In [ ]:
# ── Heatmap utilities ──────────────────────────────────────────────────────

TARGET_SHAPE = (96, 128, 128)  # fixed output size — all volumes resized to this

def make_heatmap(shape, center_zyx, sigma=3.0, downsample_factor=6):
    """3D Gaussian heatmap, peak=1 at center_zyx (original-space coords)."""
    D, H, W = shape
    cz = center_zyx[0] / downsample_factor
    cy = center_zyx[1] / downsample_factor
    cx = center_zyx[2] / downsample_factor
    z = np.arange(D, dtype=np.float32)
    y = np.arange(H, dtype=np.float32)
    x = np.arange(W, dtype=np.float32)
    dist_sq = (
        (z[:, None, None] - cz) ** 2
        + (y[None, :, None] - cy) ** 2
        + (x[None, None, :] - cx) ** 2
    )
    return np.exp(-dist_sq / (2.0 * sigma ** 2)).astype(np.float32)


def soft_argmax_3d(heatmap: torch.Tensor) -> torch.Tensor:
    """(D,H,W) → (3,) [z,y,x] in downsampled voxels via softmax."""
    D, H, W = heatmap.shape
    weights = torch.softmax(heatmap.view(-1), dim=0).view(D, H, W)
    z_g = torch.arange(D, dtype=weights.dtype, device=weights.device)
    y_g = torch.arange(H, dtype=weights.dtype, device=weights.device)
    x_g = torch.arange(W, dtype=weights.dtype, device=weights.device)
    z = (weights.sum(dim=[1, 2]) * z_g).sum()
    y = (weights.sum(dim=[0, 2]) * y_g).sum()
    x = (weights.sum(dim=[0, 1]) * x_g).sum()
    return torch.stack([z, y, x])


def coords_from_heatmap(heatmap: torch.Tensor, downsample_factor: int = 6):
    ds = soft_argmax_3d(heatmap)
    return ds, ds * float(downsample_factor)


# ── Augmentation ───────────────────────────────────────────────────────────

def _augment(volume, left_orig, right_orig):
    if random.random() < 0.7:
        volume = np.clip(random.uniform(0.9, 1.1) * volume + random.uniform(-0.05, 0.05), 0.0, 1.0)

    if random.random() < 0.4:
        volume = np.clip(volume + np.random.normal(0, 0.01, volume.shape), 0.0, 1.0).astype(np.float32)

    if random.random() < 0.5:
        volume = np.flip(volume, axis=2).copy()
        ds_W = volume.shape[2]
        left_new  = [right_orig[0], right_orig[1], (ds_W - 1 - right_orig[2] // 6) * 6]
        right_new = [left_orig[0],  left_orig[1],  (ds_W - 1 - left_orig[2] // 6) * 6]
        left_orig, right_orig = left_new, right_new

    if random.random() < 0.5:
        angle = random.uniform(-10, 10)
        axes  = random.choice([(0, 1), (0, 2), (1, 2)])
        volume = ndimage.rotate(volume, angle, axes=axes, reshape=False, order=1, mode='nearest')
        D, H, W = volume.shape
        centers = [D / 2, H / 2, W / 2]
        a, b = axes
        c = np.cos(np.radians(angle)); s = np.sin(np.radians(angle))
        dim_sizes = [D * 6, H * 6, W * 6]
        for orig in (left_orig, right_orig):
            da = (orig[a] / 6 - centers[a])
            db = (orig[b] / 6 - centers[b])
            orig[a] = int(np.clip((da * c - db * s + centers[a]) * 6, 0, dim_sizes[a] - 1))
            orig[b] = int(np.clip((da * s + db * c + centers[b]) * 6, 0, dim_sizes[b] - 1))

    return volume.astype(np.float32), left_orig, right_orig


# ── Dataset ────────────────────────────────────────────────────────────────

class TMJHeatmapDataset(Dataset):
    def __init__(self, study_ids, annotations_dir, volumes_dir,
                 sigma=3.0, downsample_factor=6, is_train=True):
        self.sigma             = sigma
        self.downsample_factor = downsample_factor
        self.is_train          = is_train
        self.ann_dir           = Path(annotations_dir)
        self.volumes_dir       = Path(volumes_dir)

        self.records = []
        for sid in study_ids:
            ann_path = self.ann_dir / f'{sid}_rois.json'
            if not ann_path.exists():
                print(f'  WARNING: annotation missing for {sid}, skipping')
                continue
            with open(ann_path) as f:
                ann = json.load(f)
            self.records.append({
                'study_id':     ann['scan_id'],
                'left_center':  list(ann['left_tmj']['center']),
                'right_center': list(ann['right_tmj']['center']),
            })
        print(f'Dataset: {len(self.records)} samples ({"train" if is_train else "val"})')

    def _load_volume(self, study_id: str) -> np.ndarray:
        npy = self.volumes_dir / f'{study_id}.npy'
        if not npy.exists():
            raise FileNotFoundError(f'Volume not found: {npy}')
        vol = np.array(np.load(str(npy)), dtype=np.float32) / 255.0
        # Resize to fixed shape if original DICOM had different dimensions
        if vol.shape != TARGET_SHAPE:
            zoom = [t / s for t, s in zip(TARGET_SHAPE, vol.shape)]
            vol = ndimage.zoom(vol, zoom, order=1)
        return vol.astype(np.float32)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec   = self.records[idx]
        vol   = self._load_volume(rec['study_id'])
        left  = list(rec['left_center'])
        right = list(rec['right_center'])

        if self.is_train:
            vol, left, right = _augment(vol, left, right)

        ds_shape = vol.shape
        hm_left  = make_heatmap(ds_shape, left,  self.sigma, self.downsample_factor)
        hm_right = make_heatmap(ds_shape, right, self.sigma, self.downsample_factor)

        # torch.tensor() copies data → tensors own their storage
        vol_t = torch.tensor(vol, dtype=torch.float32).unsqueeze(0)
        hm_t  = torch.tensor(np.stack([hm_left, hm_right]), dtype=torch.float32)
        return vol_t, hm_t


# ── Loss ───────────────────────────────────────────────────────────────────

def weighted_mse_loss(pred, target, pos_weight=10.0):
    weight = 1.0 + pos_weight * target
    return (weight * (pred - target) ** 2).mean()


print('Utilities defined')

In [ ]:
# ── Hyperparams ────────────────────────────────────────────────────────────
SIGMA             = 3.0
DOWNSAMPLE_FACTOR = 6
BATCH_SIZE        = 4    # V100 32GB handles 4 easily; try 6 if no OOM
NUM_WORKERS       = 0    # pin_memory + num_workers > 0 crashes on PyTorch 2.0.x
LR                = 1e-4
WEIGHT_DECAY      = 1e-4
EPOCHS            = 200
LR_PATIENCE       = 15
EARLY_STOP        = 40

# Fixed input size → cuDNN can find optimal conv algorithms once and reuse
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True

train_ds = TMJHeatmapDataset(split['train'], ANN_DIR, VOLUMES_DIR,
                              sigma=SIGMA, downsample_factor=DOWNSAMPLE_FACTOR, is_train=True)
val_ds   = TMJHeatmapDataset(split['val'],   ANN_DIR, VOLUMES_DIR,
                              sigma=SIGMA, downsample_factor=DOWNSAMPLE_FACTOR, is_train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=False)

print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

## 5. Training

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=LR_PATIENCE)
scaler    = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None

ts      = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
exp_dir = EXP_DIR / f'heatmap_{ts}'
exp_dir.mkdir(parents=True, exist_ok=True)

config = dict(sigma=SIGMA, downsample_factor=DOWNSAMPLE_FACTOR,
               batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY,
               epochs=EPOCHS, lr_patience=LR_PATIENCE, early_stopping=EARLY_STOP,
               heatmap=True, timestamp=ts)
with open(exp_dir / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f'Experiment: {exp_dir}')


def compute_mae_batch(pred_hm: torch.Tensor, target_hm: torch.Tensor, ds_factor: int = 6) -> float:
    """MAE in downsampled px. pred_hm/target_hm: (B,2,D,H,W) float32 CPU tensors."""
    pred_hm   = pred_hm.float()
    target_hm = target_hm.float()
    B = pred_hm.shape[0]
    errs = []
    for b in range(B):
        for ch in range(2):
            pc, _ = coords_from_heatmap(torch.sigmoid(pred_hm[b, ch]), ds_factor)
            tc, _ = coords_from_heatmap(target_hm[b, ch], ds_factor)
            errs.append(torch.sqrt(((pc - tc) ** 2).sum()).item())
    return float(np.mean(errs))


def run_epoch(model, loader, optimizer, is_train, epoch_num):
    model.train() if is_train else model.eval()
    total_loss = 0.0
    all_mae    = []   # only populated for val

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for vols, targets in tqdm(loader, desc=f'[{epoch_num}] {"Train" if is_train else "Val  "}', leave=False):
            vols, targets = vols.to(device), targets.to(device)

            if is_train:
                optimizer.zero_grad()
                if scaler:
                    with torch.cuda.amp.autocast():
                        pred = model(vols)
                        loss = weighted_mse_loss(torch.sigmoid(pred), targets)
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    pred = model(vols)
                    loss = weighted_mse_loss(torch.sigmoid(pred), targets)
                    loss.backward()
                    optimizer.step()
                # Skip MAE during training — expensive CPU soft-argmax, not needed per batch
            else:
                pred = model(vols)
                loss = weighted_mse_loss(torch.sigmoid(pred), targets)
                # MAE only on val — drives scheduler + early stopping
                all_mae.append(compute_mae_batch(pred.detach().cpu(), targets.cpu(), DOWNSAMPLE_FACTOR))

            total_loss += loss.item()

    result = {'loss': total_loss / len(loader)}
    if all_mae:
        result['mae'] = float(np.mean(all_mae))
    return result


print('Training setup done')

In [ ]:
best_mae = float('inf')
no_imp   = 0

print(f'{"Ep":>4}  {"tr_loss":>8}  │  {"va_loss":>8}  {"va_mae":>7}')
print('─' * 42)

for epoch in range(1, EPOCHS + 1):
    tr  = run_epoch(model, train_loader, optimizer, True,  epoch)
    val = run_epoch(model, val_loader,   optimizer, False, epoch)
    scheduler.step(val['mae'])
    lr_now = optimizer.param_groups[0]['lr']

    line = (f"{epoch:4d}  {tr['loss']:8.4f}  │  "
            f"{val['loss']:8.4f}  {val['mae']:7.2f}  lr={lr_now:.1e}")

    if val['mae'] < best_mae:
        best_mae = val['mae']
        no_imp   = 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'best_val_mae': best_mae, 'config': config},
                   exp_dir / 'best_model.pth')
        line += f'  ✓ best ({best_mae:.2f} ds_px ≈ {best_mae * DOWNSAMPLE_FACTOR:.0f} orig_px ≈ {best_mae * DOWNSAMPLE_FACTOR * 0.4:.1f} mm)'
    else:
        no_imp += 1

    print(line)

    with open(exp_dir / 'metrics.jsonl', 'a') as f:
        f.write(json.dumps({'epoch': epoch, 'lr': lr_now,
                            'train_loss': tr['loss'],
                            'val_loss': val['loss'], 'val_mae': val['mae']}) + '\n')

    if EARLY_STOP > 0 and no_imp >= EARLY_STOP:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nDone. Best val MAE: {best_mae:.2f} ds_px ≈ {best_mae * DOWNSAMPLE_FACTOR:.0f} orig_px '
      f'≈ {best_mae * DOWNSAMPLE_FACTOR * 0.4:.1f} mm')
print(f'Checkpoint: {exp_dir / "best_model.pth"}')

## 6. Upload checkpoint to GitHub Release (optional)

In [ ]:
# Run this cell only if you want to upload the checkpoint back to GitHub.
# Requires GITHUB_TOKEN set as an env variable or DataSphere secret.

import subprocess, os

GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')
REPO         = 'tzopiz/MasterProject'
RELEASE_TAG  = f'heatmap-detector-{ts}'
CKPT_PATH    = exp_dir / 'best_model.pth'

if not GITHUB_TOKEN:
    print('GITHUB_TOKEN not set — skipping upload')
else:
    # Create release
    import urllib.request, json as _json
    headers = {'Authorization': f'token {GITHUB_TOKEN}',
               'Content-Type': 'application/json',
               'Accept': 'application/vnd.github.v3+json'}
    body = _json.dumps({'tag_name': RELEASE_TAG, 'name': f'Heatmap Detector {ts}',
                         'body': f'best_val_mae={best_mae:.2f}ds_px'}).encode()
    req = urllib.request.Request(
        f'https://api.github.com/repos/{REPO}/releases', data=body,
        headers=headers, method='POST')
    with urllib.request.urlopen(req) as resp:
        rel = _json.loads(resp.read())
    upload_url = rel['upload_url'].split('{')[0]

    # Upload checkpoint
    with open(CKPT_PATH, 'rb') as f:
        data = f.read()
    req2 = urllib.request.Request(
        f'{upload_url}?name=best_model.pth', data=data,
        headers={**headers, 'Content-Type': 'application/octet-stream'}, method='POST')
    with urllib.request.urlopen(req2) as resp2:
        asset = _json.loads(resp2.read())
    print(f'Uploaded: {asset["browser_download_url"]}')